# ShopDesk, Module 2 Section 2 Lab 1: Configuring MCP Servers

A beginner-friendly notebook on wiring **MCP servers** into ShopDesk. First the **Claude
Code** config workflow (`.mcp.json`, `~/.claude.json`, and `${VAR}` credentials) that you edit
in a Git repo, then a runnable **Claude Agent SDK** setup that connects **two servers at once**
and reads a credential from the environment. Runs **Sonnet** (`claude-sonnet-4-6`) through your
**Anthropic API key**.

## The real-world scenario

ShopDesk's data lives in several systems: an orders database, a shipping API, a returns
service. **MCP** lets each become a set of tools the agent can call, and `.mcp.json` is how you
declare them so the whole team shares one config. The trick is doing it **securely**: the config
is committed to Git, but the secrets are not; they come from environment variables at connect
time.

The question this lab answers: **how do you declare multiple MCP servers, keep their credentials
out of source control, and make all their tools available at once?**

## Objectives

- Configure MCP servers in **`.mcp.json`** (project, version-controlled) and **`~/.claude.json`**
  (user).
- Reference credentials with **`${VAR}`** so secrets stay in the environment, not in Git.
- Connect **multiple servers** so all their tools are available simultaneously, and reference
  **MCP resources** to load context on demand.

## What you'll observe

- A pure-Python expander resolves `${VAR}` and `${VAR:-default}` against the environment.
- A validator flags a config that hardcodes a secret and passes one that uses `${VAR}`.
- Live, one Agent SDK options object exposes tools from two in-process servers at the same time.

## How to run

The `.mcp.json` section is **terminal/config** reference: edit those files in a real repo with
Claude Code installed. The Python cells (expander, validator, multi-server setup) run here. The
multi-server run calls Claude, so paste a real key into **Setup 2/3**; otherwise it skips.
**Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages for the runnable Agent SDK section. Claude Code itself
is installed separately in your terminal for the config section. The Agent SDK also needs Node.js
18+, which cannot be pip-installed.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` so the async
multi-server run can be called like a normal function.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read env vars (this lab is all about env vars)
import re                                       # expand ${VAR} in configs
import sys                                       # detect Windows (it needs a special event loop)
import json                                     # build and print configs
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the agent will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

### MCP servers, and where they are configured

An **MCP server** exposes capabilities through three primitives: **tools** (actions the agent can
call), **resources** (read-only data the agent can load on demand), and **prompts** (reusable
slash-command templates). Claude Code reads server configuration from two places:

- **`.mcp.json`** at the project root: shared with the team and committed to Git, so everyone gets
  the same servers.
- **`~/.claude.json`** (user level): your personal servers, available in every project.

Secrets never go in either file directly. You reference them with `${VAR}`, and Claude Code
expands them from your shell at connect time.

**This section is config**, meant for a real repo (not run by the notebook). A single
`.mcp.json` can declare several servers, and all their tools become available at once. Note the
`${VAR}` and `${VAR:-default}` references: the config is safe to commit because the secrets live in
the environment.

```json
{
  "mcpServers": {
    "orders-db": {
      "type": "stdio",
      "command": "npx",
      "args": ["-y", "@shopdesk/orders-mcp", "--dsn", "${ORDERS_DSN}"],
      "env": { "LOG_LEVEL": "info" }
    },
    "shipping-api": {
      "type": "http",
      "url": "${SHIPPING_BASE_URL:-https://api.ship.example.com}/mcp",
      "headers": { "Authorization": "Bearer ${SHIPPING_API_KEY}" }
    },
    "github": {
      "type": "http",
      "url": "https://api.githubcopilot.com/mcp/",
      "headers": { "Authorization": "Bearer ${GITHUB_TOKEN}" }
    }
  }
}
```

From the CLI you can add the same servers without editing JSON by hand:

```bash
claude mcp add --transport http --scope project shipping-api \
  "https://api.ship.example.com/mcp" --header "Authorization: Bearer ${SHIPPING_API_KEY}"
claude mcp add --transport stdio --scope project orders-db \
  -- npx -y @shopdesk/orders-mcp --dsn "${ORDERS_DSN}"
```

Check status any time with `/mcp`. A user-level server goes in `~/.claude.json` instead of the
project file so it loads everywhere.

---

### 🎯 Lab objective - declare servers, keep secrets safe, load many at once

**What you build:** a `.mcp.json` in Python, an env-var expander and a validator that both run
offline, and a live two-server Agent SDK setup.

**Why it helps you build real solutions:** teams share one config but must not share secrets, and
real agents pull from several systems. Getting the `${VAR}` pattern and multi-server wiring right is
the foundation.

**How you'll see it:** the expander resolves references, the validator catches a hardcoded secret,
and one options object exposes tools from two servers.

**This cell:** builds the same `.mcp.json` as a Python dict and prints it. Generating config
in code is handy for scripts and tests; the shape is exactly what Claude Code reads.

In [ ]:
# ===== build a .mcp.json (as a Python dict) =====
MCP_CONFIG = {                                     # the config Claude Code would read
    "mcpServers": {
        "orders-db": {"type": "stdio", "command": "npx",
                      "args": ["-y", "@shopdesk/orders-mcp", "--dsn", "${ORDERS_DSN}"]},
        "shipping-api": {"type": "http",
                         "url": "${SHIPPING_BASE_URL:-https://api.ship.example.com}/mcp",
                         "headers": {"Authorization": "Bearer ${SHIPPING_API_KEY}"}},
    }
}
print(json.dumps(MCP_CONFIG, indent=2))            # this is what you would commit to Git

**This cell:** an **env-var expander**, pure Python. It resolves `${VAR}` from the
environment and `${VAR:-default}` with a fallback, leaving an unset, no-default reference as a
literal (which Claude Code would warn about). This shows exactly how a committed config becomes a
live one at connect time.

In [ ]:
# ===== expand ${VAR} and ${VAR:-default} like Claude Code does =====
def expand(value):                                 # expand env references inside one string
    def repl(m):                                   #   handle one ${...} match
        var, default = m.group(1), m.group(3)      #     the name and optional default
        val = os.environ.get(var)                  #     look it up in the environment
        if val is not None: return val             #     found -> use it
        if default is not None: return default     #     not found -> use the default if any
        return m.group(0)                          #     neither -> leave the literal (a warning case)
    return re.sub(r"\$\{([A-Za-z_][A-Za-z0-9_]*)(:-([^}]*))?\}", repl, value)

def expand_config(obj):                            # walk the config and expand every string
    if isinstance(obj, dict):  return {k: expand_config(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [expand_config(v) for v in obj]
    if isinstance(obj, str):   return expand(obj)
    return obj

os.environ["SHIPPING_API_KEY"] = "shpk_demo_123"   # pretend this came from your shell / secret store
print("SHIPPING_API_KEY set, SHIPPING_BASE_URL and ORDERS_DSN unset:\n")
print(json.dumps(expand_config(MCP_CONFIG), indent=2))   # key filled, url uses default, dsn stays literal

**This cell:** a small **validator**. It checks each server has a `command` or a `url`, and,
crucially, flags any value that looks like a **hardcoded secret** instead of a `${VAR}` reference.
Catching a committed secret before it reaches Git is the whole point of the `${VAR}` pattern.

In [ ]:
# ===== validate a config: structure and no hardcoded secrets =====
SECRET_LOOKS_LIKE = re.compile(r"(ghp_|sk-ant-|shpk_|AKIA)[A-Za-z0-9_-]{6,}")   # rough secret shapes

def validate(cfg):                                 # config dict -> list of problems
    issues = []                                     #   collect findings
    if "mcpServers" not in cfg:                     #   the top-level key must exist
        return ["missing 'mcpServers'"]
    for name, s in cfg["mcpServers"].items():       #   check each server
        if "command" not in s and "url" not in s:   #     needs one transport
            issues.append(f"{name}: needs a 'command' (stdio) or 'url' (http)")
        if SECRET_LOOKS_LIKE.search(json.dumps(s)): #     a raw secret slipped in?
            issues.append(f"{name}: HARDCODED SECRET - use ${{VAR}} instead")
    return issues or ["ok: structure valid, no hardcoded secrets"]

print("good config:", validate(MCP_CONFIG))         # uses ${VAR} -> clean
bad = {"mcpServers": {"x": {"type": "http", "url": "https://a", "headers": {"Authorization": "Bearer shpk_live_ABCDEF"}}}}
print("bad config: ", validate(bad))                # hardcoded secret -> flagged

**This cell:** **MCP resources**, which optimize workflows by loading context only when it is
referenced. Unlike tools (actions), a resource is read-only data (a schema, a file tree, a record).
In Claude Code you pull one into the prompt with an `@` reference, so the agent gets exactly the
context it needs and nothing more.

In [ ]:
# ===== MCP resources: load context on demand =====
print("tool     -> an action the agent calls, e.g. mcp__orders-db__get_order")
print("resource -> read-only context, referenced with @, e.g. @orders-db:schema://orders")
print("prompt   -> a reusable template, e.g. /mcp__orders-db__triage_ticket")
print("Referencing @orders-db:record://A1 loads just that order's context, not the whole DB.")

**This cell:** the runnable **multi-server** setup. We build two in-process servers (orders
and shipping) with `create_sdk_mcp_server`, and the shipping tool reads its credential from an
**environment variable** rather than a literal. Both servers go into one options object, so all
their tools are available at once.

In [ ]:
# ===== two in-process MCP servers, wired together =====
from claude_agent_sdk import query, ClaudeAgentOptions, tool, create_sdk_mcp_server, AssistantMessage, ResultMessage, TextBlock, ToolUseBlock

ORDERS = {"A1": {"status": "shipped"}, "A2": {"status": "delivered"}}   # the orders server's data

@tool("get_order", "Get an order's status by id.", {"order_id": str})
async def get_order(args):                          # a tool on the ORDERS server
    o = ORDERS.get(args["order_id"], {"status": "unknown order"})
    return {"content": [{"type": "text", "text": o["status"]}]}
orders_srv = create_sdk_mcp_server(name="orders", version="1.0.0", tools=[get_order])

SHIPPING_KEY = os.environ.get("SHIPPING_API_KEY", "demo-key")   # credential FROM the environment

@tool("get_tracking", "Get the tracking link for a shipped order.", {"order_id": str})
async def get_tracking(args):                       # a tool on the SHIPPING server
    return {"content": [{"type": "text",             #   in real life it would auth with SHIPPING_KEY
            "text": f"https://track.example/{args['order_id']} (auth={SHIPPING_KEY[:4]}...)"}]}
shipping_srv = create_sdk_mcp_server(name="shipping", version="1.0.0", tools=[get_tracking])

MULTI = ClaudeAgentOptions(                          # ONE options object, BOTH servers
    model=MODEL,
    mcp_servers={"orders": orders_srv, "shipping": shipping_srv},
    allowed_tools=["mcp__orders__get_order", "mcp__shipping__get_tracking"])
print("connected servers:", list(MULTI.mcp_servers), "-> all tools available at once")

**This cell:** runs one request that needs **both** servers: the order's status (orders
server) and its tracking link (shipping server). A single agent reaches across both, which is what
"all MCP tools accessible simultaneously" means in practice.

In [ ]:
# ===== run a request that spans both servers =====
async def ask(prompt):                              # stream one query and print tool calls + answer
    answer = ""
    async for m in query(prompt=prompt, options=MULTI):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  -> tool:", b.name.split("__")[-1], b.input)
                elif isinstance(b, TextBlock):  answer = b.text
    print("ANSWER:", answer)

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: ask("What is order A1's status, and what is its tracking link?"))
else:
    print("[skipped] expected: get_order (orders server) AND get_tracking (shipping server)")
    print("          both fire in one run, from the two connected servers.")

| anti-pattern | what to do instead |
|---|---|
| paste an API key straight into `.mcp.json` | reference it as `${VAR}` and keep the secret in your shell |
| commit `.env` or a filled config to Git | commit the `${VAR}` config; document required vars in the README |
| one giant server for everything | split by system (orders, shipping) and connect several at once |
| load a whole dataset as a tool result | expose it as a resource and reference just what you need with `@` |

**Lesson:** `.mcp.json` (project) and `~/.claude.json` (user) declare your MCP servers, and
`${VAR}` keeps their secrets in the environment so the config is safe to share. Connect several
servers and all their tools are available at once; expose read-only context as resources and pull
it in on demand with `@`.

---

## Recap - configure servers safely

| Piece | In this lab | Course topic |
|---|---|---|
| `.mcp.json` | project config, committed to Git | project-level, version-controlled config |
| `~/.claude.json` | user-level servers everywhere | user-level setup |
| `${VAR}` | credentials from the environment | secure, portable credentials |
| Multi-server | two servers in one options object | all MCP tools accessible simultaneously |
| Resources | `@server:resource://...` | load context on demand |

One principle to carry forward: **declare servers in shared config, keep secrets in `${VAR}`, and
connect as many as the job needs.** To run the multi-server cell live, paste a real key into
**Setup 2/3** and re-run from the top. Then try it: add a third in-process server and expose its
tool alongside the others. Next lab: structured MCP errors and retry logic.